# Text Classification Pipeline
This notebook demonstrates a complete text classification workflow, including data cleaning, vectorization, model training, and evaluation. Each step is explained with practical notes and intuitions.

## What is Text Classification?
Text classification is the task of assigning predefined categories (labels) to text data. Common applications include spam detection, sentiment analysis, and topic labeling.
- **Supervised**: Requires labeled data.
- **Pipeline**: Data → Clean → Vectorize → Model → Predict → Evaluate.

In [ ]:
# 1. Imports
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

## Step 1: Data Loading
For demonstration, we use a small sample dataset. Replace this with your own data for real projects.

In [ ]:
# 2. Sample Data
data = {
    'text': [
        'I love this product!',
        'This is the worst experience ever',
        'Absolutely fantastic service',
        'I hate it',
        'Not bad, could be better',
        'Terrible, will not buy again'
    ],
    'label': [1, 0, 1, 0, 1, 0]  # 1 = Positive, 0 = Negative
}
df = pd.DataFrame(data)
df.head()

### Notes:
- Labels must be numeric for most ML models.
- Always inspect your data for class balance and quality.

In [ ]:
# 3. Text Cleaning Function
def clean_text(text):
    """
    Basic text cleaning:
    - Lowercasing
    - Remove URLs
    - Remove punctuation/numbers
    - Remove extra spaces
    """
    text = text.lower()
    text = re.sub(r'http\S+', '', text)  # remove URLs
    text = re.sub(r'[^a-z\s]', '', text)  # keep only alphabets
    text = re.sub(r'\s+', ' ', text).strip()  # remove extra spaces
    return text
# Apply cleaning
df['clean_text'] = df['text'].apply(clean_text)
df[['text', 'clean_text']]

### Notes:
- Cleaning reduces noise and standardizes text.
- Always adapt cleaning steps to your domain (e.g., keep numbers for product reviews).

In [ ]:
# 4. Train-Test Split
# Stratify ensures class balance in both sets (important for imbalanced data).
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)
print('Train size:', len(X_train), '| Test size:', len(X_test))

### Notes:
- Never train and test on the same data!
- Stratification is crucial for fair evaluation, especially with few samples.

In [ ]:
# 5. Vectorization (TF-IDF)
# Converts text to numeric features for ML models.
vectorizer = TfidfVectorizer(
    max_features=5000,      # limit vocab size
    ngram_range=(1, 2)      # use unigrams + bigrams
)
# Fit only on training data (avoids leakage)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)
print('Vocabulary size:', len(vectorizer.get_feature_names_out()))

### Notes:
- TF-IDF weighs words by importance, reducing the impact of common words.
- n-grams capture short phrases and context.
- Always fit vectorizer on training data only to prevent information leakage.

In [ ]:
# 6. Model Training
# Logistic Regression is a strong baseline for text classification.
model = LogisticRegression(class_weight='balanced')  # class_weight helps with imbalance
model.fit(X_train_vec, y_train)

### Notes:
- Logistic Regression is fast, interpretable, and often competitive with more complex models.
- Use class_weight='balanced' for imbalanced datasets.
- Try other models (SVM, Random Forest, XGBoost) for better performance.

In [ ]:
# 7. Predictions
y_pred = model.predict(X_test_vec)
y_proba = model.predict_proba(X_test_vec)[:, 1]  # probability of class 1
print('Predictions:', y_pred)
print('Probabilities:', y_proba)

### Notes:
- Always check both predicted labels and probabilities.
- Probabilities are useful for threshold tuning and ROC analysis.

In [ ]:
# 8. Evaluation
print('Classification Report:')
print(classification_report(y_test, y_pred))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

### Notes:
- Precision: How many selected items are relevant?
- Recall: How many relevant items are selected?
- F1-score: Harmonic mean of precision and recall.
- Confusion matrix shows true/false positives/negatives.
- For imbalanced data, focus on recall or F1-score.

In [ ]:
# 9. Custom Prediction Function
def predict_text(text):
    '''Takes raw text → returns prediction + probability'''
    cleaned = clean_text(text)
    vec = vectorizer.transform([cleaned])
    pred = model.predict(vec)[0]
    prob = model.predict_proba(vec)[0][1]
    return {'prediction': int(pred), 'probability': float(prob)}
# Test with new input
sample = 'This product is amazing!'
result = predict_text(sample)
print('Sample Prediction:', result)

### Notes:
- Always preprocess new/unseen text the same way as training data.
- Use probability for thresholding or ranking predictions.

## Final Tips
- Try more data, better cleaning, and advanced models for improved results.
- For deep learning, use embeddings (Word2Vec, BERT) instead of TF-IDF.
- Always validate on real, unseen data before deploying.